In [ ]:
# Personal parameters required by HW0/HW1 standing instructions
SID4 = 1384
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10
print({"SID4": SID4, "SEED": SEED, "SLICE": SLICE, "HP_ID": HP_ID, "CLS_A": CLS_A, "CLS_B": CLS_B})

# HW2.5 — GPU Assignment I (documented working copy)

This notebook mirrors `HW2_5_GPU_Assignment.ipynb` but adds a markdown documentation cell after every measurement
section, plus two runs the original notebook did not perform: an actual attention OOM boundary search (Part D)
and a lower-precision (FP8) attempt (Part B). Fill in every `TODO` in the markdown cells directly from the outputs
printed below them — do not copy numbers from a different run or from a spec sheet.

Run this on the reserved RTX workstation with outputs intact. It does not overwrite the original notebook.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import torch

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
torch.manual_seed(SEED)

sys.path.insert(0, str(ROOT))
import benchmark_hw2_5 as bm

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## Part A — Onboarding and provenance

Capture `nvidia-smi -q` before any benchmark runs. Record the reservation and GPU-hours separately in
`RESERVATION_RECORD.md` — that data lives in your lab's reservation system, not in anything this notebook can measure.

In [ ]:
smi = subprocess.run(['nvidia-smi', '-q'], capture_output=True, text=True, check=True)
print(smi.stdout)
(RESULTS / 'nvidia-smi-q.txt').write_text(smi.stdout, encoding='utf-8')

In [ ]:
meta = bm.gpu_metadata()
props = torch.cuda.get_device_properties(0)
print('UUID:', meta['uuid'])
print('Name:', meta['name'])
print('Total memory (GiB):', meta['total_memory_bytes'] / 2**30)
print('Compute capability:', meta['compute_capability'])
print('Torch:', meta['torch_version'], '| CUDA runtime:', meta['cuda_runtime'])

### Part A documentation (fill in from the two cells above and from vendor docs)

| Field | Value | Source |
|---|---|---|
| GPU model | TODO | `results/nvidia-smi-q.txt` |
| GPU UUID | TODO | printed above |
| Driver version | TODO | `results/nvidia-smi-q.txt` |
| CUDA version | TODO | printed above |
| VRAM capacity | TODO | printed above |
| Reported power limit | TODO | `results/nvidia-smi-q.txt` |
| Architecture | TODO | vendor URL + access date |
| Memory type and bandwidth (spec) | TODO | vendor URL + access date |
| Tensor core generation | TODO | vendor URL + access date |
| Reduced precisions supported | TODO | vendor URL + access date |

## Part B — Precision and achieved throughput

FP32 / TF32 / FP16 / BF16 at N = 1024, 4096, 8192, 16384, warmed up and averaged over `repetitions`.

In [ ]:
args = argparse_ns = type('Args', (), {'sizes': bm.DEFAULT_SIZES, 'repetitions': 10})()
bm.precision_benchmark(args)
rows = bm.read_jsonl(RESULTS / 'precision.jsonl')
for row in rows[-16:]:
    print(row['precision'], row['size'], round(row['achieved_tflops'], 2), 'TFLOPS')

### Lower-precision (FP8) attempt

The assignment requires either a real FP8 measurement or a documented account of what was tried and how it failed.
The cell below attempts `torch._scaled_mm` with `float8_e4m3fn` inputs and records success or the exact failure.

In [ ]:
fp8_result = {'uuid': bm.gpu_uuid()}
try:
    a = torch.randn(1024, 1024, device='cuda').to(torch.float8_e4m3fn)
    b = torch.randn(1024, 1024, device='cuda').to(torch.float8_e4m3fn)
    scale_a = torch.tensor(1.0, device='cuda')
    scale_b = torch.tensor(1.0, device='cuda')
    seconds = bm.timed_cuda(lambda: torch._scaled_mm(a, b, out_dtype=torch.float16, scale_a=scale_a, scale_b=scale_b), repetitions=10)
    fp8_result.update({'status': 'success', 'size': 1024, 'seconds_per_matmul': seconds,
                        'achieved_tflops': 2.0 * 1024**3 / seconds / 1e12})
except Exception as error:
    fp8_result.update({'status': 'failed', 'error_type': type(error).__name__, 'error_message': str(error)})
print(fp8_result)
(RESULTS / 'fp8_attempt.json').write_text(json.dumps(fp8_result, indent=2), encoding='utf-8')

In [ ]:
bm.plot_results(None)
print('Wrote results/precision_tflops.png')

### Part B documentation (fill in from the outputs above)

- State the plateau size for each precision and why small matrices never reach peak throughput (launch/occupancy overhead): TODO
- Achieved TFLOPS as a percentage of theoretical peak, per precision (cite the vendor peak number from Part A): TODO
- FP8 result: TODO — either report the achieved TFLOPS, or quote the `error_type`/`error_message` from `fp8_attempt.json` and explain what tooling gap it reveals.

## Part C — Bandwidth-bound vs compute-bound

Elementwise add (memory-bound) and a large square matmul (compute-bound).

In [ ]:
args = type('Args', (), {'elements': 256 * 1024 * 1024, 'matmul_size': 8192, 'repetitions': 10})()
bm.bandwidth_benchmark(args)
for row in bm.read_jsonl(RESULTS / 'roofline.jsonl')[-2:]:
    print(row['kind'], {k: v for k, v in row.items() if k in ('effective_bandwidth_gb_s', 'achieved_tflops', 'arithmetic_intensity_flops_per_byte')})

### Part C documentation

- Effective bandwidth achieved vs. the card's specified bandwidth (from Part A): TODO
- Arithmetic intensity of each op (FLOPs/byte, printed above) and which side of the roofline each falls on for this card's ridge point (peak TFLOPS / peak bandwidth): TODO

## Part D — The cost of attention

Naive (materialized N×N attention matrix) vs. fused `scaled_dot_product_attention`, head_dim=128, batch=1,
at lengths 512–16384, then an actual OOM boundary search beyond that range for both implementations.

In [ ]:
args = type('Args', (), {'sizes': bm.ATTENTION_SIZES, 'head_dim': 128, 'batch': 1, 'repetitions': 3})()
bm.attention_benchmark(args)
for row in bm.read_jsonl(RESULTS / 'attention.jsonl')[-12:]:
    print('fused' if row['fused'] else 'naive', row['length'], row['peak_memory_bytes'] / 2**20, 'MiB', row['oom'])

### OOM boundary search

Exponential search to find a failing length, then binary search to single-token resolution between the largest
length that succeeded and the smallest that failed. This is the step the original run skipped — it never went
far enough to actually hit an OOM for either implementation.

In [ ]:
def find_oom_boundary(fused, head_dim=128, batch=1, start=16384):
    length = start
    largest_ok = None
    smallest_fail = None
    while smallest_fail is None:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            bm.attention_once(length, head_dim, batch, fused)
            torch.cuda.synchronize()
            largest_ok = length
            length *= 2
        except RuntimeError as error:
            if 'out of memory' not in str(error).lower():
                raise
            smallest_fail = length
            torch.cuda.empty_cache()
    lo, hi = largest_ok, smallest_fail
    while hi - lo > 1:
        mid = (lo + hi) // 2
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            bm.attention_once(mid, head_dim, batch, fused)
            torch.cuda.synchronize()
            lo = mid
        except RuntimeError as error:
            if 'out of memory' not in str(error).lower():
                raise
            hi = mid
            torch.cuda.empty_cache()
    return lo, hi

boundary_rows = []
for fused in (False, True):
    largest_ok, smallest_fail = find_oom_boundary(fused)
    print(('fused' if fused else 'naive'), 'largest_ok =', largest_ok, '| smallest_fail =', smallest_fail)
    boundary_rows.append({**bm.gpu_metadata(), 'fused': fused, 'head_dim': 128, 'batch': 1,
                           'largest_ok_length': largest_ok, 'smallest_fail_length': smallest_fail})

bm.append_jsonl(RESULTS / 'attention_oom_boundary.jsonl', boundary_rows)

In [ ]:
bm.plot_results(None)
print(Path('results/attention_fit.txt').read_text())

### Part D documentation

- Naive OOM boundary — largest length that succeeded / smallest that failed: TODO (from `attention_oom_boundary.jsonl`)
- Fused OOM boundary — largest length that succeeded / smallest that failed: TODO
- Fitted quadratic coefficient from `attention_fit.txt`, and confirmation this matches the O(N²) expectation: TODO
- Fused speedup at each tested length (naive seconds / fused seconds): TODO
- Three or four sentences on what the fused kernel avoids materializing: TODO

## Part E — Sustained load and thermal behaviour

20-minute sustained matmul load, sampled every 5 seconds. Do not shorten `--duration` for the graded run.

In [ ]:
args = type('Args', (), {'duration': 1200, 'interval': 5, 'matmul_size': 8192})()
bm.thermal_benchmark(args)
bm.plot_results(None)
print('Wrote results/thermal.csv and results/thermal_clock_temperature.png')

### Part E documentation

- Did throttling occur? At what temperature/power ceiling, and how many seconds into the run: TODO
- Peak throughput in the first 30s vs. steady-state throughput in the final 5 minutes, as a percentage: TODO

In [ ]:
for figure in RESULTS.glob('*.png'):
    shutil.copy2(figure, FIGURES / figure.name)
print('Figures:', sorted(p.name for p in FIGURES.glob('*')))
print('Raw results:', sorted(p.name for p in RESULTS.glob('*')))

## Part F — Summary table

Transcribe the final numbers into `METRICS.md` Table HW2.5.1, and record every command run in this notebook
(with exact start/end UTC and the reservation ID) as its own block in `RUN_LOG.txt`. Update `RESERVATION_RECORD.md`
with actual start/end and consumed GPU-hours, and `AI_USE.md` with what AI assistance was and was not used for.
Then `git init` (if not already), commit, and tag the commit `hw2-5`.

| Measurement | Your GPU | Notes |
|---|---:|---|
| Peak achieved TFLOPS (BF16) | TODO | UUID + `results/precision.jsonl` |
| % of theoretical peak (BF16) | TODO | cite vendor peak and method |
| Effective bandwidth (GB/s) | TODO | UUID + `results/roofline.jsonl` |
| Naive attention OOM length | TODO | UUID + `results/attention_oom_boundary.jsonl` |
| Fused attention OOM length | TODO | UUID + `results/attention_oom_boundary.jsonl` |
| Steady-state / peak throughput | TODO | UUID + `results/thermal.csv` |
| Throttle onset (s, or none) | TODO | UUID + `results/thermal.csv` |